## Wrapping up Week 4: Generating iambs

Last week, we worked through several steps towards creating lines in iambic pentameter.  Along the way we introduced the following Python concepts:

- List slicing
- Defining a function
- The `in` and `not` operators
- The string method `.join()` and the list method `.append()`

The steps we are following to generate iambic pentameter:

1. Create a list of words from which to idetify iambs to use
    a) open and read a file
    b) clean the data to remove punctuation
    c) split the text into a list of words
2. Write a function to look up the pronunciation of a word and return its stress pattern
3. Create a list of words that are iambs by comparing their stress pattern to "01"
4. Randomly select 5 words at a time from our list of iambs to generate a lines of iambic pentameter.

Spoiler: This isn't a very satisfying approach, so we will look at one way to enhance this approach.

In [ ]:
import re

# open the file "Shakespeare_Sonnets.txt" for reading and store its contents as 
# a list of lines in the variable lines
with open("Shakespeare_Sonnets.txt", encoding="utf-8") as f:
    lines = f.readlines()

raw_lines = lines[34:2654]

poem_text_lines = []
for line in raw_lines:
    stripped_line = line.strip()

    if not stripped_line:  # line is empty
        continue
    # If stripped is all upper case then skip it.
    elif stripped_line == stripped_line.upper():
        continue
    else:
        # remove punctuation (except apostrophes) and set to lower case
        cleaned_line = re.sub(r"[^a-z'\u2019]", " ", line.lower())
        poem_text_lines.append(cleaned_line)

print(len(poem_text_lines), "usable lines of verse")
print(poem_text_lines[100])

# Also add the get_pronunciation function
import nltk
nltk.download('cmudict')
from nltk.corpus import cmudict
# Create a Python dictionary from the cmudict and store it in pronounce_dictionary
pronounce_dictionary = cmudict.dict()

def get_pronunciation(word):
    ''' Return the first phoneme list found for the word word
    '''
    word = word.lower()

    if word in pronounce_dictionary:
        return pronounce_dictionary[word][0]
    else:
        return None

In [ ]:
# Last week we wrote this function
def get_stress_pattern(word):
    ''' Return the stress pattern as a string for the word given as an argument.
        Example: get_stress_pattern("awake") returns "01"
    '''
    # the body of the function is indented
    stress_pattern = "" # empty string to start
    word_p = get_pronunciation(word)

    # If there isn't a pronunciation then return None
    if not word_p:
        return None

    for phoneme in word_p:
        last_character = phoneme[-1]
        if last_character.isdigit():
            stress_pattern = stress_pattern + last_character
    return stress_pattern

Now we are at step 3: creating a list of words that are iambs, but first we need to split the Shakespeare poem lines into a list of unique words.  

(This is a code pattern we will see frequently.)

In [ ]:
# Create an empty list to store unique words from poem_text_lines
word_bank = []
for line in poem_text_lines:
    # split the line into a list of words (splitting on whitespace)
    line_words = line.split()

    for word in line_words:
        if word not in word_bank:
            word_bank.append(word)

# How many words are in our word bank?

print(len(word_bank)) 

We can finally create a list of iambs!!

In [ ]:
iambs = []

for word in word_bank:

    pattern = get_stress_pattern(word)

    if pattern == "01":
        iambs.append(word)
    if pattern == "11":
        iambs.append(word)

print(len(iambs))

# What does the next line do?
iambs[:20]

In [ ]:
# And finally, randomly select 5 iambs per line 
import random

for i in range(0, 14):
    line_chunks = random.sample(iambs, 5)
    print(" ".join(line_chunks))

### Three new things

1. Opening a file and reading lines from the file into a list
2. Instead of randomly selecting one item at time from a list, `random.sample(list, num)` allows us to select `num` things from the list at a time.
3. `str.join(list)`

Most often, we will use `str.join(list)` to insert spaces or commas between list elements.

In [ ]:
# An example to illustrate the .join() method

word_list = ["Here", "is", "a", "silly", "example."]

# Create a string by joining the words in word_list with " "
print(" ".join(word_list))

# Create a string by joining the words in word_list with " like "
joiner = " like "
print(joiner.join(word_list))



# Week 5: From Poems to Metadata — Reading Tabular Data

New Python concepts:
- Reading a file line by line
- Comma-separated values (CSV) files and the `csv` module
- Data cleaning  (next week)
- Writing a file (next week)
- Dictionaries

## 2. What is a CSV file?

A spreadsheet program such as Excel or Google Sheets displays data as a table of **rows** and **columns**. One of the simplest ways to store a spreadsheet in a file is the **comma-separated values** (CSV) format:

- Each line of the file is one **row** of the spreadsheet.
- Within a line, the values for each **column** are separated by commas.
- Usually the first line is a **header** that gives a name to each column.



Here are the first three lines of `giller_nominees.csv`, exactly as they appear in the file:

```
Year ,Title ,Author ,Publisher ,Publisher Location ,Born ,DOB,Residence ,DOD,Bio,Novel
1994,Funny Boy,Shyam Selvadurai,,,"Colombo, Sri Lanka",12-02-1965,"Toronto, ON",00/00/00,https://thecanadianencyclopedia.ca/...
1994,The Museum of Love,Steve Weiner,,,"Milwaukee, WI",1947,"London, England ",21/04/2024,https://en.wikipedia.org/wiki/Steve_Weiner,
```

Take a minute to look at these lines with a neighbour. What do you notice?

## 3. Reading a file, line by line

In [ ]:
# The "with" block makes sure the file is closed again when we are done with it.
# encoding="utf-8" tells Python how the characters in the file are stored.

with open("giller_nominees_clean.csv", encoding="utf-8") as f:
    # .readlines() reads the whole file and returns a list of strings,
    # one string per line of the file.
    lines = f.readlines()

print(len(lines), "lines in the file")

`lines` is a list, so everything we know about lists still works. Line 0 is the header, and each line after that is one row of the csv file, or one nominated book.

In [ ]:
print(lines[0])
print(lines[1])
print(lines[-1])

We could go deeper into using the Python we already know to split each row into columns, but there is a module that works with CSV files that will make our life easier.

In [ ]:
import csv

with open("giller_nominees.csv", encoding="utf-8") as f:
    reader = csv.reader(f)
    rows = list(reader)   # turn all of the rows into one big list

print(len(rows), "rows")
print(rows[2])

Now we can work with different columns.

In [ ]:
header = rows[0]
print(header)

book = rows[2]
print(book[1], "by", book[2], "born in", book[5])

This is a little hard to work with. To get a birthplace we write `book[5]`, but six months from now, will you remember that column 5 means "birthplace"? Wouldn't it be nicer to write `book["Born"]`?

To do that we need a new kind of value: the **dictionary**.

## Dictionaries

A Python **dictionary** works a lot like a real dictionary. In a real dictionary you look up a *word* to find its *definition*. In a Python dictionary you look up a **key** to find its **value**.

- Python dictionaries 
    - look up a value using a key
    - `empty_dict = {}`
    - `short_dict = {"Karen" : "CS", "Nat : "English"}`
- Python list 
    - look up a value using an index 
    - `empty_list = []`
    - `short_list = ["Karen", "Nat"]`

In [ ]:
winners = {
    "The Book of Secrets": "M.G. Vassanji",
    "A Fine Balance": "Rohinton Mistry",
    "Alias Grace": "Margaret Atwood",
}

print(winners)


In [ ]:
print(type(winners))

In [ ]:

print(winners.keys())

In [ ]:

print(winners.values())

### Looking up a value

To look up a value we put the key in square brackets, just like indexing a list. With a list, we use a *position* number; with a dictionary, we use the *key*.

In [ ]:
winners["Alias Grace"]

What happens if we look up a key that isn't in the dictionary?

In [ ]:
winners["Fall on Your Knees"]

A `KeyError` means "there is no such key." We can check first with the `in` operator, which we met last week. With a dictionary, `in` checks the **keys**.

In [ ]:
title = "Fall on Your Knees"

if title in winners:
    print(f"{title} was written by {winners[title]}")
else:
    print(f"{title} is not in our dictionary")

### Adding and changing values

We add a new key–value pair, or change an existing one, with an assignment statement. If the key is already in the dictionary, its old value is replaced. If not, the pair is added.

In [ ]:
winners["How to Pronounce Knife"] = "Souvankham Thammavongsa"
print(winners)
print(len(winners), "entries")

**Keys are unique.** A dictionary can't contain the same key twice. Assigning to an existing key *replaces* its value. This is often exactly what we want, as we'll see when we start counting.



### Looping over a dictionary

A `for` loop over a dictionary visits each **key**. Then we can use the key to look up its value.

In [ ]:
for title in winners:
    print(f"{title} ({winners[title]})")

## Each row as a dictionary: `csv.DictReader`

Let's return to the question of how to write `book["Born"]`. The `csv` module has a second tool, `csv.DictReader()`. It uses the header line as **keys** and turns every row into a dictionary.

In [ ]:
with open("giller_nominees_clean.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    nominees = list(reader)

print(len(nominees), "nominated books")   # notice the header is not counted as a book
print(nominees[1])

This makes it more legible to identify different aspects of the data we have about individual books.

In [ ]:
book = nominees[1]
print(book["Author"])
print(f"{book['Title']} by {book['Author']} ({book['Year']})")

Now we can start to take a look at this data in more interesting ways. Let's print all of the books from one year:

In [ ]:
# for each row of the table stored in nominees
for book in nominees:
    if book["Year"] == "1996":
        print(f"{book['Title']} by {book['Author']}")

## How many times was each author nominated?

Now we can answer our first real question. We want a new table with one row per **author**, giving the number of times they were nominated. A dictionary is perfect for this: the **keys** will be author names and the **values** will be counts.

This is one of the most useful patterns and is a variation on a pattern we have already seen:
- Start with an empty dictionary.
- For each book, look at its author.
    - If we've seen this author before, add 1 to their count.
    - If not, this is their first nomination, so set their count to 1.

In [ ]:
# create an empty dictionary
nomination_counts = {}

# Go through the rows in the table
for book in nominees:
    # set author to the author of the current row
    author = book["Author"]

    # if we have already started counting nominations for author
    # then we look up their nomination count so far and add one to it
    # otherwise, this is the first nomination we have see for this author
    # so we set its value in the dictionary to 1.
    if author in nomination_counts:
        nomination_counts[author] = nomination_counts[author] + 1
    else:
        nomination_counts[author] = 1

print(len(nomination_counts), "different authors")

In [ ]:
print(nomination_counts["Margaret Atwood"])
print(nomination_counts["Wayne Johnston"])